# 03. 판정 함수 (2026-11-17, 2026-11-18 D57, 2026-11-23 D64·D65 반영)

- 규칙은 `src/rules/`(rules.csv, d43_range.csv, transfer_terms.csv), 엔진은 `src/judge.py`, 조건 함수는 `src/predicates.py`.
- 이 노트북은 함수를 불러 쓰고, 기존 결과(제외 364, 교정 533, tags.csv 21행, D19)가 그대로 재현되는지 확인한다.
- 대상: 증권 CoT 7,201건 (D18). 판정 없이 사실만 출력한다. 원문 문장은 결과 파일에 넣지 않는다.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
from src.load_cot import load_cot, load_products
from src.judge import judge, load_rules, load_range

AS_OF = "2026-11-24"  # D57(2026-11-18), ①②·P5(2026-11-23, D64·D65), R0·사람확인 판정(2026-11-24, D67) 포함
cot = load_cot(); prod = load_products()
print("시행 중 허용 범위(공개기준):", {p: sorted(g) for p, g in load_range(AS_OF).items()})
rules = load_rules(AS_OF)
print("규칙 수:", len(rules)); print(rules.groupby("기준").size().to_string())

In [ ]:
j, ctx = judge(cot, prod, as_of=AS_OF)
print("판정 행 수:", len(j), j["분할"].value_counts().to_dict())

## 1. 기존 결과 재현 확인

하나라도 다르면 assert에서 멈춘다.
D65(2026-11-23, P5 우선)로 새로 제외되는 CoT는 exclusions.csv에 근거 D65로 더한다 (다시 돌리면 D65 행을 지우고 다시 쓴다).

In [ ]:
ex = pd.read_csv(ROOT / "outputs" / "exclusions.csv", encoding="utf-8-sig")
ex = ex[ex["근거 결정"] != "D65"]
new = j[(j["기준③ 처리"] == "제외") & j["P5 우선"] & ~j["cot_id"].isin(ex["cot_id"])]
ex = pd.concat([ex, pd.DataFrame({"cot_id": new["cot_id"], "분할": new["분할"],
                                  "제외 사유": "P5 우선: 복수 성향 중 더 낮은 성향(P8)으로 범위 밖", "근거 결정": "D65", "출처": "D65"})],
               ignore_index=True)
ex.to_csv(ROOT / "outputs" / "exclusions.csv", index=False, encoding="utf-8-sig")
print("[D65] exclusions.csv에 더한 CoT:", len(new), sorted(new["cot_id"]))
a, b = set(j.loc[j["기준③ 처리"] == "제외", "cot_id"]), set(ex["cot_id"])
print("[제외] judge:", len(a), "/ exclusions.csv:", len(b), "/ judge에만:", sorted(a - b), "/ csv에만:", sorted(b - a))
assert a == b

log = pd.read_csv(ROOT / "outputs" / "corrections_log.csv", encoding="utf-8-sig", dtype=str)
print("[교정] judge:", j["교정"].sum(), "/ corrections_log 행:", len(log), "(고유 cot", log["cot_id"].nunique(), ")")
assert set(j.loc[j["교정"], "cot_id"].astype(str)) == set(log["cot_id"])

tags = pd.read_csv(ROOT / "outputs" / "tags.csv", encoding="utf-8-sig", dtype=str)
jt = {(str(c), t.strip()) for c, s in zip(j["cot_id"], j["기록 태그"]) if s for t in s.split(",")}
print("[tags.csv] judge 기록 태그 쌍:", len(jt), "/ tags.csv 행:", len(tags))
assert jt == set(zip(tags["cot_id"], tags["태그"]))

d19 = j["기준④ 판정"].value_counts()
print("[D19]", d19.to_dict())
assert (d19["전부 기재"], d19["일부 기재"], d19["기재 없음"]) == (1478, 1614, 4105)
print("모두 재현됨")

## 2. 기준별 결과 (분할별)

In [ ]:
for col in ["기준③ 판정", "사유 코드", "D30 태그", "기준④ 판정", "원천 확인 필요", "원천 등급 복수", "처리"]:
    t = pd.crosstab(j[col].replace("", "(없음)"), j["분할"], margins=True, margins_name="합계")
    print(f"[{col}]"); print(t.to_string()); print()

In [ ]:
# 태그 건수 (tags.csv 밖에서 규칙이 만든 태그 포함)
tag_counts = {
    "D19 일부 기재": (j["기준④ 판정"] == "일부 기재").sum(),
    "D19 기재 없음": (j["기준④ 판정"] == "기재 없음").sum(),
    "D30 완화된 허용 범위": (j["D30 태그"] != "").sum(),
    "이관 결함 (D50)": (j["이관 판정"] == "결함").sum(),
    "tags.csv 기록 태그 CoT": (j["기록 태그"] != "").sum(),
}
print(pd.Series(tag_counts).to_string())

## 3. 원천 확인 필요 (D51) → B안 근거 첨부 대상

- 등록 등급과 설명 속 등급(body_grade)이 다른 상품을 추천한 CoT. 상품 목록은 `ctx["mismatch_keys"]`.

In [ ]:
src_cots = j[j["원천 확인 필요"]]
print("원천 확인 필요 CoT:", len(src_cots), src_cots["분할"].value_counts().to_dict(),
      "/ 그중 제외:", (src_cots["기준③ 처리"] == "제외").sum())
rec = ctx["recs"]
hit = rec[rec["keys"].map(lambda ks: any(k in ctx["mismatch_keys"] for k in ks))]
print("대상 상품(추천된 것):", hit["rec_name"].nunique(), "/ 불일치 상품 전체:", len(ctx["mismatch_keys"]))

## 4. 이관 규칙 (D50·D56)

- 분류 순서: 회피 → 경험 → R4 비교 표현 → R1(숫자 등급 + 요청) → R2(고려 표현) → 사람 확인. 키워드는 `src/rules/transfer_terms.csv`.
- 이관 표현: 같은 문장에 범위 초과 고지와 상담·확인 절차가 함께 있을 때만 (D56). 일반 상담 문구는 이관으로 보지 않는다.
- '사람 확인'은 키워드를 늘리지 않고 그대로 남긴다 (C안: LLM 1차 판정 대상).

In [ ]:
print(pd.crosstab(j["이관 분류"], j["이관 판정"], margins=True, margins_name="합계"))
print()
mention = j["이관 분류"] != "언급 없음"
outside = j["성향 밖 언급"] != ""
h = j["이관 판정"] == "사람 확인"
print("등급 언급 CoT:", mention.sum(), "/ 성향 밖 언급 CoT:", outside.sum(), "/ 성향 불명 + 언급:", (h & ~outside).sum())
print(f"사람 확인: {h.sum()}건 (등급 언급 CoT 대비 {h.sum() / mention.sum() * 100:.1f}%)")
print(f"  성향 밖 언급 CoT 중 사람 확인: {(h & outside).sum()} / {outside.sum()} ({(h & outside).sum() / outside.sum() * 100:.1f}%)")
print("사람 확인 사유:"); print(j.loc[h, "사람 확인 사유"].value_counts().to_string())
print()
print("결함:", (j["이관 판정"] == "결함").sum(), j.loc[j["이관 판정"] == "결함", "분할"].value_counts().to_dict(),
      "/ 그중 기준③ 제외:", ((j["이관 판정"] == "결함") & (j["기준③ 처리"] == "제외")).sum())
print("R3 대상(성향 밖 희망인데 그 등급 경험 문장 없음):", j["R3 대상"].sum())

In [ ]:
# 이관 표현과 일반 상담 문구 (D56)
print("이관 표현 있는 CoT:", j["이관 표현"].sum())
print("일반 상담 문구 있는 CoT:", j["일반 상담 문구"].sum(), j.loc[j["일반 상담 문구"], "분할"].value_counts().to_dict())
print("  그중 성향 밖 언급 CoT:", (j["일반 상담 문구"] & outside).sum(), "/ 이관 결함 CoT:", (j["일반 상담 문구"] & (j["이관 판정"] == "결함")).sum())

## 5. 희망검수표 1·2 (40건)와 대조

- 규칙 키워드는 이 40건을 보고 정했으므로 일치율이 실제보다 높게 나온다. 새 표본(이관검수표 20건)은 04에서 대조한다.
- 대조 단위: 사람이 본 언급 등급에 대한 규칙 분류. 희망검수표1의 '오탐(일반 표현)'에 맞는 규칙 분류는 없다.

In [ ]:
from src.transfer import load_terms, question_mentions
terms = load_terms()
qtext = cot.set_index("cot_id")["question"]
rows = []
for f in ["희망검수표", "희망검수표2"]:
    d = pd.read_csv(ROOT / "data" / "review" / f"{f}.csv", encoding="utf-8-sig", dtype=str, keep_default_na=False)
    for _, x in d.iterrows():
        qm = question_mentions(qtext[int(x["cot_id"])], terms)
        m = qm[qm["언급 등급"] == int(x["언급 등급"])]
        rows.append({"표": f, "cot_id": int(x["cot_id"]), "사람": x["내 판정"],
                     "규칙 분류": m["분류"].iloc[0] if len(m) else "언급 못 찾음"})
cmp = pd.DataFrame(rows).merge(j[["cot_id", "이관 판정"]], on="cot_id")
print(pd.crosstab(cmp["사람"], cmp["규칙 분류"], margins=True))
print()
print(pd.crosstab(cmp["사람"], cmp["이관 판정"], margins=True))
MAP = {"회피": "회피", "희망(명확)": "R1 명확한 희망", "고려 중": "R2 고려 중", "경험": "경험", "비교 표현": "R4 비교 표현"}
ok = cmp["사람"].map(MAP) == cmp["규칙 분류"]
print()
print(f"분류 일치 (오탐 제외 {cmp['사람'].isin(MAP).sum()}건): {ok.sum()}건")
print("희망검수표2 결함 일치 (사람: 20건 모두 결함):", (cmp.loc[cmp["표"] == "희망검수표2", "이관 판정"] == "결함").sum(), "/ 20")
print(cmp[~ok].to_string(index=False))

## 6. 판정 결과 저장 → `outputs/judgments.csv`

CoT당 한 행. 원문 문장 없음.

In [ ]:
out_path = ROOT / "outputs" / "judgments.csv"
j.to_csv(out_path, index=False, encoding="utf-8-sig")
print("저장:", out_path, j.shape)
print(list(j.columns))

## 7. 04 검증용 이관 검수표

- 04 노트북(`notebooks/04_검증.ipynb`)으로 옮겼다 (2026-11-17, 성향 밖 15 + 성향 안 5로 다시 뽑음).